# 16 · Time-Series Backtesting

## What this notebook covers
Evaluating models on time-series data requires backtesting strategies that respect temporal order. Standard k-fold CV on time-series data produces leakage (future data in training) and violates the i.i.d. assumption. This notebook covers the three canonical backtesting schemes and when to use each.

## Why standard CV fails on time series
Cross-validation assumes exchangeable samples. Time-series data has autocorrelation: knowing yesterday's value tells you something about today's. Shuffling breaks the temporal structure and leaks future information into training windows.

## The three backtesting schemes
| Scheme | Training window | Use when |
|--------|----------------|---------|
| **Expanding window** | Grows with each fold | You want to use all historical data |
| **Sliding window** | Fixed size, shifts forward | Stationarity assumption holds; old data is not informative |
| **Walk-forward** | Retrain at every step | High-frequency, drift-prone data |

## Horizon matters
A model that predicts 1 step ahead is much easier to evaluate than one that predicts 12 steps ahead. Backtesting should match the deployment horizon — if your model is used for 30-day forecasts, backtest over 30-day horizons.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

# ── Synthetic time series: trend + seasonality + noise ─────────────────────────
n = 500
t = np.arange(n)
trend = 0.05 * t
seasonal = 10 * np.sin(2 * np.pi * t / 52)  # weekly cycle
noise = np.random.randn(n) * 3
y_ts = trend + seasonal + noise

# Feature engineering: lags + time features
def make_features(y, lags=[1, 2, 3, 7, 14, 28], horizon=1):
    df = pd.DataFrame({"y": y})
    for lag in lags:
        df[f"lag_{lag}"] = df["y"].shift(lag)
    df["t"] = np.arange(len(y))
    df["sin_52"] = np.sin(2 * np.pi * df["t"] / 52)
    df["cos_52"] = np.cos(2 * np.pi * df["t"] / 52)
    df = df.dropna()
    X = df.drop(columns=["y"]).values
    y_out = df["y"].values
    return X, y_out

X_ts, y_ts_clean = make_features(y_ts)
print(f"Feature matrix: {X_ts.shape}")

In [ ]:
def expanding_window_backtest(X, y, min_train=100, horizon=1, step=20):
    """
    Expanding window: training set grows at each step.
    Returns list of (train_end_idx, mae) tuples.
    """
    n = len(X)
    results = []
    for end in range(min_train, n - horizon, step):
        X_tr, y_tr = X[:end], y[:end]
        X_te_w = X[end:end+horizon]
        y_te_w = y[end:end+horizon]
        clf = GradientBoostingRegressor(n_estimators=100, random_state=0)
        clf.fit(X_tr, y_tr)
        results.append({"fold_end": end, "mae": mean_absolute_error(y_te_w, clf.predict(X_te_w)), "scheme": "expanding"})
    return results

def sliding_window_backtest(X, y, window=150, horizon=1, step=20):
    """
    Sliding window: fixed-size training window.
    """
    n = len(X)
    results = []
    for end in range(window, n - horizon, step):
        X_tr, y_tr = X[end-window:end], y[end-window:end]
        X_te_w = X[end:end+horizon]
        y_te_w = y[end:end+horizon]
        clf = GradientBoostingRegressor(n_estimators=100, random_state=0)
        clf.fit(X_tr, y_tr)
        results.append({"fold_end": end, "mae": mean_absolute_error(y_te_w, clf.predict(X_te_w)), "scheme": "sliding"})
    return results

expanding = expanding_window_backtest(X_ts, y_ts_clean, min_train=100, horizon=7, step=20)
sliding   = sliding_window_backtest(X_ts, y_ts_clean, window=150, horizon=7, step=20)

exp_mae = np.mean([r["mae"] for r in expanding])
sld_mae = np.mean([r["mae"] for r in sliding])
print(f"Expanding window mean MAE: {exp_mae:.4f}")
print(f"Sliding window mean MAE  : {sld_mae:.4f}")

In [ ]:
# Horizon sensitivity: how does MAE degrade with forecast horizon?
horizons = [1, 3, 7, 14, 21, 28]
horizon_maes = []
for h in horizons:
    folds = sliding_window_backtest(X_ts, y_ts_clean, window=150, horizon=h, step=30)
    horizon_maes.append(np.mean([r["mae"] for r in folds]))
    print(f"Horizon={h:2d}: MAE={horizon_maes[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Expanding vs sliding MAE over time
exp_df = pd.DataFrame(expanding)
sld_df = pd.DataFrame(sliding)
axes[0].plot(exp_df["fold_end"], exp_df["mae"], label="Expanding", color="steelblue")
axes[0].plot(sld_df["fold_end"], sld_df["mae"], label="Sliding",   color="tomato")
axes[0].set_xlabel("Fold end index"); axes[0].set_ylabel("MAE")
axes[0].set_title("MAE across backtesting folds (h=7)")
axes[0].legend()

# Horizon sensitivity
axes[1].plot(horizons, horizon_maes, "o-", color="steelblue", lw=2)
axes[1].set_xlabel("Forecast horizon (steps)")
axes[1].set_ylabel("Mean MAE")
axes[1].set_title("MAE degradation with forecast horizon")
axes[1].axhline(horizon_maes[0], color="gray", linestyle="--", lw=1)

# Time series with train/test split illustrated
split = 350
axes[2].plot(y_ts[:split], color="steelblue", lw=1.5, label="Train")
axes[2].plot(range(split, len(y_ts)), y_ts[split:], color="tomato", lw=1.5, label="Test")
axes[2].axvline(split, color="black", linestyle="--", lw=1)
axes[2].set_title("Synthetic time series (train/test)")
axes[2].legend()

plt.tight_layout()
plt.savefig(f"{base}/16_timeseries_backtesting.png", dpi=120)
plt.show()

## Summary
- Never shuffle time-series data for cross-validation
- **Expanding window** is the default for non-stationary series (uses all history)
- **Sliding window** is preferred when old data becomes uninformative (e.g., regime change)
- Always backtest at the **deployment horizon** — a 30-day forecast model should be evaluated with 30-step-ahead errors, not 1-step
- `TimeSeriesSplit` in sklearn implements expanding window; sliding window requires a custom loop as shown above